# 11 - Data dictionary
Documents every table: column, data type, % missing, whether the value is known before the match, and a description.
It produces docs/data_dictionary.md (saved in the bucket and downloaded, so it can be committed to the repo), and writes the descriptions into BigQuery so they show next to the data in the console.
Run after 10_split. Re-run whenever a table changes.

## Install pinned libraries
Read directly from the Github repo

In [ ]:
# Read from a git tag rather than from main, so a later re-run installs exactly the same versions.
CODE_VERSION = "code-v1"      # the git tag of the code being run: create it on GitHub before running
REPO = "katiedevane/tennis-project"
REQUIREMENTS_URL = f"https://raw.githubusercontent.com/{REPO}/{CODE_VERSION}/requirements.txt"
!pip install -q -r $REQUIREMENTS_URL

# Shared provenance helpers (code version, environment, index of data versions), from the same tag
PROVENANCE_URL = f"https://raw.githubusercontent.com/{REPO}/{CODE_VERSION}/provenance.py"
!wget -q -O provenance.py $PROVENANCE_URL

## Authenticate and setup
Enable Notebook access for both secrets in this notebook.

In [ ]:
import os
from google.colab import auth, userdata

auth.authenticate_user()

PROJECT_ID = userdata.get("TENNIS_PROJECT_ID")
BUCKET_NAME = userdata.get("TENNIS_BUCKET_NAME")
REGION = "EU"        # the bucket and the BigQuery datasets must share this location
DATASETS = ["tennis_raw", "tennis_clean", "tennis_features"]
os.environ["CLOUDSDK_CORE_PROJECT"] = PROJECT_ID

assert PROJECT_ID and BUCKET_NAME, "A secret is empty: check its name and Notebook access"
print("Settings loaded from Colab Secrets.")

## Record the code version and set the random seed
The git tag, commit, Python environment and random seed of this run are added to every manifest this notebook writes (see provenance.py).
`SEED` is the one project-wide seed: `start_run` applies it, and it should be passed wherever a function takes a seed (e.g. `random_state=SEED`).

In [ ]:
from provenance import start_run, register, table_version, sha256, utc_stamp, SEED

NOTEBOOK = "11_data_dictionary"
RUN_INFO = start_run(f"https://github.com/{REPO}", CODE_VERSION, NOTEBOOK)

## Helpers

In [ ]:
import pandas as pd
from google.cloud import bigquery, storage

bq = bigquery.Client(project=PROJECT_ID, location=REGION)
bucket = storage.Client(project=PROJECT_ID).bucket(BUCKET_NAME)

def run(sql):
    config = bigquery.QueryJobConfig(maximum_bytes_billed=1_000_000_000)   # safety cap: 1 GB per query
    return bq.query(sql, job_config=config).to_dataframe()

pd.set_option("display.max_rows", 300, "display.max_colwidth", 120, "display.width", 200)

## Descriptions
Written once per column name. Columns that repeat for each player (`winner_`/`loser_`, `a_`/`b_`, `w_`/`l_`, `post_a_`/`post_b_`) are described from their base name.
`known_before_match` marks what a model may use: `no (post-match)` columns can only feed features built from *earlier* matches.

In [ ]:
TABLE_NOTES = {
    "tennis_raw.wta_matches": "Raw WTA matches, every column STRING (external table over the compressed CSV in the bucket). Typed in 04_clean_matches.",
    "tennis_clean.matches": "Typed matches, levels G/PM/P/I, no round-robin events. One row per match, winner/loser layout.",
    "tennis_clean.matches_dated": "tennis_clean.matches plus the event format and a window of possible match days.",
    "tennis_clean.venue_name_map": "Tournament name key -> place searched for in the geocoding API.",
    "tennis_clean.venue_locations": "One row per place: coordinates and the geocoding result chosen.",
    "tennis_clean.tournament_editions": "One row per tournament edition, with coordinates and the weather window fetched.",
    "tennis_clean.matches_ab": "One row per match in player A vs player B layout, label a_won. Columns starting post_ are only known after the match.",
    "tennis_features.match_splits": "Split per match (by season), and whether the row can be used for modelling.",
}

# Exact column names
COLUMNS = {
    "match_id": ("Match ID: tourney_id + '-' + match_num. Unique per match", "id"),
    "tourney_id": ("Tournament edition ID: season year + tournament code", "yes"),
    "tourney_name": ("Tournament name as in the source", "yes"),
    "name_key": ("Lower-cased, trimmed tourney_name; join key to the venue tables", "yes"),
    "tourney_date": ("Tournament start date (raw: text YYYYMMDD). Not the match date", "yes"),
    "season": ("Season = year in tourney_id. Late-December events belong to the next season", "yes"),
    "tourney_level": ("Level: G = Grand Slam, PM = Premier Mandatory, P = Premier, I = International. Raw also has D, O, W, 35+H, 50+H (excluded in 04)", "yes"),
    "surface": ("Court surface: hard, clay, grass (lower-cased from 04 on)", "yes"),
    "draw_size": ("Number of players in the main draw", "yes"),
    "best_of": ("Maximum number of sets", "yes"),
    "round": ("Round: R128, R64, R32, R16, QF, SF, F (RR = round robin, excluded in 04)", "yes"),
    "match_num": ("Match number within the tournament", "id"),
    "score": ("Final score text; contains RET (retired), W/O (walkover), DEF (default)", "no (post-match)"),
    "minutes": ("Match length in minutes (recorded from 2016 on)", "no (post-match)"),
    "is_walkover": ("TRUE if the score contains W/O (no tennis played)", "no (post-match)"),
    "is_retirement": ("TRUE if the score contains RET", "no (post-match)"),
    "is_default": ("TRUE if the score contains DEF", "no (post-match)"),
    "event_format": ("SLAM, TWO_WEEK, LONG_WEEK or ONE_WEEK (from level and the first round played)", "yes"),
    "day_min": ("Earliest day (offset from tourney_date) the match could have been played", "yes (estimate)"),
    "day_max": ("Latest day (offset from tourney_date) the match could have been played", "yes (estimate)"),
    "match_date_min": ("Earliest possible match date = tourney_date + day_min", "yes (estimate)"),
    "match_date_max": ("Latest possible match date = tourney_date + day_max", "yes (estimate)"),
    "search_name": ("Place name searched for in the geocoding API", "yes"),
    "place": ("Place name returned by the geocoding API", "yes"),
    "region": ("Region / state of the place (geocoding admin1)", "yes"),
    "country_code": ("Country of the place (ISO 3166-1 alpha-2)", "yes"),
    "latitude": ("Latitude of the place, degrees", "yes"),
    "longitude": ("Longitude of the place, degrees", "yes"),
    "elevation_m": ("Elevation of the place, metres", "yes"),
    "timezone": ("IANA time zone of the place", "yes"),
    "geonames_id": ("GeoNames ID of the chosen result", "id"),
    "population": ("Population of the place (geocoding API)", "yes"),
    "decision": ("How the result was chosen: top result, re-searched, or picked by hand", "metadata"),
    "start_date": ("Weather window start = tourney_date", "yes"),
    "end_date": ("Weather window end = last possible match day + 1, cut back for the archive delay", "yes"),
    "last_match_day": ("Latest possible match day of the edition", "yes (estimate)"),
    "n_matches": ("Number of matches in the edition", "metadata"),
    "a_won": ("LABEL: 1 if player A won, 0 otherwise", "label"),
    "post_score": ("Final score text (see score)", "no (post-match)"),
    "post_minutes": ("Match length in minutes (from 2016 on)", "no (post-match)"),
    "post_is_walkover": ("TRUE if walkover (no tennis played); such rows are excluded from modelling", "no (post-match)"),
    "post_is_retirement": ("TRUE if a player retired", "no (post-match)"),
    "post_is_default": ("TRUE if a player was defaulted", "no (post-match)"),
    "split": ("warmup (2010-15), train (2016-22), dev (2023-24), test (2025), holdout_2026", "metadata"),
    "modelling_ok": ("FALSE for walkovers: kept, but excluded from fitting and evaluation", "metadata"),
    "split_version": ("Version of the split definition", "metadata"),
}

# Fields that repeat per player, by base name
PLAYER = {
    "id": "player ID (stable across seasons)",
    "name": "player name",
    "seed": "seed number (NULL = unseeded; raw may hold entry codes)",
    "entry": "entry: Q qualifier, WC wildcard, LL lucky loser, PR protected ranking, SE special exempt, ALT alternate; NULL = direct entry",
    "hand": "playing hand: R, L, U = unknown",
    "ht": "height, cm",
    "ioc": "country, 3-letter IOC code",
    "age": "age in years at tourney_date",
    "rank": "WTA ranking at tourney_date",
    "rank_points": "WTA ranking points at tourney_date",
}
STATS = {
    "ace": "aces", "df": "double faults", "svpt": "serve points played", "1stIn": "first serves in",
    "1stWon": "points won on first serve", "2ndWon": "points won on second serve",
    "SvGms": "service games played", "bpSaved": "break points saved", "bpFaced": "break points faced",
}
PREFIXES = [("post_a_", "Player A", STATS), ("post_b_", "Player B", STATS),
            ("winner_", "Winner", PLAYER), ("loser_", "Loser", PLAYER),
            ("a_", "Player A", PLAYER), ("b_", "Player B", PLAYER),
            ("w_", "Winner", STATS), ("l_", "Loser", STATS)]


def describe(col):
    """(description, known before the match?) for a column name."""
    if col in COLUMNS:
        return COLUMNS[col]
    for prefix, who, fields in PREFIXES:
        base = col[len(prefix):]
        if col.startswith(prefix) and base in fields:
            if fields is STATS:
                return f"{who}: {fields[base]}", "no (post-match)"
            if who in ("Winner", "Loser"):
                return f"{who}: {fields[base]}", "yes, but the winner/loser side reveals the result"
            return f"{who}: {fields[base]}", "yes"
    return "TODO: add a description", "?"

## Build the dictionary
Types are read from the live BigQuery schema, so they cannot drift from the data.

In [ ]:
frames = []
for table in TABLE_NOTES:
    t = bq.get_table(f"{PROJECT_ID}.{table}")
    names = [f.name for f in t.schema]
    raw = table.startswith("tennis_raw")      # raw columns are text: empty text counts as missing too
    missing = lambda n: f"`{n}` IS NULL OR `{n}` = ''" if raw else f"`{n}` IS NULL"
    pct = run("SELECT " + ", ".join(f"ROUND(100 * COUNTIF({missing(n)}) / COUNT(*), 1) AS `{n}`" for n in names)
              + f" FROM `{PROJECT_ID}.{table}`").T[0]
    for f in t.schema:
        desc, timing = describe(f.name)
        frames.append({"table": table, "column": f.name, "type": f.field_type, "pct_missing": pct[f.name],
                       "known_before_match": timing, "description": desc})

dictionary = pd.DataFrame(frames)
print("columns documented:", len(dictionary), "| without a description:", (dictionary["description"].str.startswith("TODO")).sum())
dictionary[dictionary["description"].str.startswith("TODO")]

## Write the descriptions into BigQuery
They then show in the console schema view of each table.

In [ ]:
for table in TABLE_NOTES:
    t = bq.get_table(f"{PROJECT_ID}.{table}")
    d = dictionary[dictionary["table"] == table].set_index("column")
    t.schema = [bigquery.SchemaField(f.name, f.field_type, mode=f.mode, fields=f.fields,
                                     description=(d.loc[f.name, "description"] + f" | known before match: {d.loc[f.name, 'known_before_match']}")[:1024])
                for f in t.schema]
    t.description = t.description or TABLE_NOTES[table]
    try:
        bq.update_table(t, ["schema", "description"])
        print("updated:", table)
    except Exception as e:
        print("not updated:", table, "|", repr(e)[:150])

## Save docs/data_dictionary.md
Saved in the bucket and downloaded. Commit the downloaded file to the repo under `docs/`.

In [ ]:
lines = [f"# Data dictionary", "",
         f"Generated by 11_data_dictionary from the live BigQuery schemas | code {RUN_INFO['code_version']} ({RUN_INFO['code_commit'][:8]}) | {utc_stamp()}", ""]
for table, note in TABLE_NOTES.items():
    d = dictionary[dictionary["table"] == table]
    lines += [f"## {table}", "", note, "", "| column | type | % missing | known before match | description |", "|---|---|---|---|---|"]
    lines += [f"| `{r.column}` | {r.type} | {r.pct_missing} | {r.known_before_match} | {r.description} |" for r in d.itertuples()]
    lines += [""]
text = "\n".join(lines)

bucket.blob("docs/data_dictionary.md").upload_from_string(text, content_type="text/markdown")
with open("data_dictionary.md", "w") as f:
    f.write(text)

from google.colab import files
files.download("data_dictionary.md")
print(text[:1500])